# TESS exoplanet classification — Dual-Branch 1D ResNet

A reproducible baseline that combines sector-detrended flux/centroid sequences with stellar, statistical, centroid-correlation, and Box Least Squares features. It uses four camera/CCD-grouped folds, saves the best checkpoint for each fold, reports out-of-fold macro-F1, and averages fold probabilities for the test submission.

Run from the repository root with the project `data/` directory present. BLS feature extraction is the most CPU-intensive preprocessing stage; tune `BLS_N_PERIODS`, batch size, and epochs for your machine.

In [1]:
from pathlib import Path
import os
import random
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from scipy import stats
from astropy.timeseries import BoxLeastSquares
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_class_weight

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = DEVICE.type == 'cuda'
print(f'Device: {DEVICE} | mixed precision: {USE_AMP} | PyTorch: {torch.__version__}')

DATA = Path('./data')
if not DATA.is_dir():
    raise FileNotFoundError(f'Data directory not found: {DATA.resolve()}')
TRAIN_DIR, TEST_DIR = DATA / 'train', DATA / 'test'
train_flux = np.load(TRAIN_DIR / 'train_flux.npy', mmap_mode='r')
train_centroid = np.load(TRAIN_DIR / 'train_centroid.npy', mmap_mode='r')
test_flux = np.load(TEST_DIR / 'test_flux.npy', mmap_mode='r')
test_centroid = np.load(TEST_DIR / 'test_centroid.npy', mmap_mode='r')
train_stars = pd.read_csv(TRAIN_DIR / 'train_stars.csv')
test_stars = pd.read_csv(TEST_DIR / 'test_stars.csv')
train_cadences = pd.read_csv(TRAIN_DIR / 'train_cadences.csv')
test_cadences = pd.read_csv(TEST_DIR / 'test_cadences.csv')

EXPECTED_CLASSES = {
    'planet', 'eclipsing_binary', 'background_binary', 'starspots',
    'pulsating_star', 'flare_star', 'instrument_glitch', 'quiet_star',
}
if set(train_stars['label'].unique()) != EXPECTED_CLASSES:
    raise ValueError('Training labels do not match the expected eight classes.')
for split, flux, centroid, stars, cadences in (
    ('train', train_flux, train_centroid, train_stars, train_cadences),
    ('test', test_flux, test_centroid, test_stars, test_cadences),
):
    if flux.ndim != 2 or centroid.shape != (len(stars), 2, flux.shape[1]):
        raise ValueError(f'Unexpected {split} flux/centroid shape.')
    if len(stars) != flux.shape[0] or len(cadences) != flux.shape[1]:
        raise ValueError(f'{split} star/cadence rows do not match time-series dimensions.')
    if not {'camera', 'ccd'}.issubset(stars.columns):
        raise ValueError(f'{split} metadata must contain camera and ccd columns.')

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(train_stars['label'].astype(str)).astype(np.int64)
class_names = label_encoder.classes_
train_groups = (train_stars['camera'].astype(str) + '_' + train_stars['ccd'].astype(str)).to_numpy()
test_groups = (test_stars['camera'].astype(str) + '_' + test_stars['ccd'].astype(str)).to_numpy()
print('Train/test:', train_flux.shape, test_flux.shape)
print('Classes:', list(class_names))
print('Unique train camera/CCD groups:', len(np.unique(train_groups)))

Device: cuda | mixed precision: True | PyTorch: 2.14.1+cu130
Train/test: (18000, 1315) (7000, 1315)
Classes: ['background_binary', 'eclipsing_binary', 'flare_star', 'instrument_glitch', 'planet', 'pulsating_star', 'quiet_star', 'starspots']
Unique train camera/CCD groups: 16


## Sector-invariant preprocessing

Normalize each star by its median flux, remove the per-cadence median common mode within each camera/CCD group, and center centroid coordinates per star. Preprocess train and test independently because they represent different observing sectors. The flux channel is expressed in parts per thousand (ppt) for stable neural-network inputs.

In [2]:
def preprocess_sector(flux, centroid, stars):
    n_stars, n_cadences = flux.shape
    result = np.zeros((n_stars, 3, n_cadences), dtype=np.float32)
    group_keys = (stars['camera'].astype(str) + '_' + stars['ccd'].astype(str)).to_numpy()
    for group in pd.unique(group_keys):
        rows = np.flatnonzero(group_keys == group)
        raw = np.asarray(flux[rows], dtype=np.float32)
        with warnings.catch_warnings():
            warnings.simplefilter('ignore', RuntimeWarning)
            row_median = np.nanmedian(raw, axis=1, keepdims=True)
        row_median[~np.isfinite(row_median) | (np.abs(row_median) < 1e-12)] = np.nan
        normalized = raw / row_median - 1.0
        with warnings.catch_warnings():
            warnings.simplefilter('ignore', RuntimeWarning)
            common_mode = np.nanmedian(normalized, axis=0, keepdims=True)
        cleaned_flux = np.nan_to_num(normalized - common_mode, nan=0.0, posinf=0.0, neginf=0.0)
        result[rows, 0] = cleaned_flux * 1000.0

        positions = np.asarray(centroid[rows], dtype=np.float32)
        with warnings.catch_warnings():
            warnings.simplefilter('ignore', RuntimeWarning)
            center = np.nanmedian(positions, axis=2, keepdims=True)
        movement = np.nan_to_num(positions - center, nan=0.0, posinf=0.0, neginf=0.0)
        result[rows, 1:] = movement
    return result

X_ts_train = preprocess_sector(train_flux, train_centroid, train_stars)
X_ts_test = preprocess_sector(test_flux, test_centroid, test_stars)
print('Preprocessed arrays:', X_ts_train.shape, X_ts_test.shape)
print('All finite:', np.isfinite(X_ts_train).all(), np.isfinite(X_ts_test).all())

Preprocessed arrays: (18000, 3, 1315) (7000, 3, 1315)
All finite: True True


## Astronomical and tabular features

Features include robust flux statistics, correlation between flux dips and centroid motion, BLS best period/depth/SNR, stellar parameters, and camera/CCD identifiers. BLS scans a configurable frequency grid; the coarse default keeps this starter pipeline practical, and a denser grid can improve transit sensitivity.

In [3]:
from joblib import Parallel, delayed

BLS_MIN_PERIOD = 0.5
BLS_MAX_PERIOD = 20.0
BLS_N_PERIODS = 128
BLS_DURATIONS = np.array([0.06, 0.12, 0.24], dtype=float)
BLS_PERIODS = np.sort(1.0 / np.linspace(1.0 / BLS_MAX_PERIOD, 1.0 / BLS_MIN_PERIOD, BLS_N_PERIODS))
META_COLUMNS = ['tmag', 'teff', 'logg', 'radius', 'contamination', 'camera', 'ccd']

# Define a worker function for a SINGLE star
def process_single_star(i, f, movement, time):
    corr = np.nan
    best_period = np.nan
    best_depth = np.nan
    best_snr = np.nan

    valid = np.isfinite(f) & np.isfinite(movement) & np.isfinite(time)
    if valid.sum() >= 3 and np.std(f[valid]) > 0 and np.std(movement[valid]) > 0:
        corr = np.corrcoef(-f[valid], movement[valid])[0, 1]
        
    bls_valid = np.isfinite(f) & np.isfinite(time)
    if bls_valid.sum() >= 20 and np.std(f[bls_valid]) > 0:
        try:
            power = BoxLeastSquares(time[bls_valid], f[bls_valid]).power(
                BLS_PERIODS, BLS_DURATIONS, objective='snr'
            )
            score = np.asarray(power.power, dtype=float)
            if score.size and np.isfinite(score).any():
                j = int(np.nanargmax(score))
                best_period = float(power.period[j])
                best_depth = float(power.depth[j])
                best_snr = float(power.depth_snr[j])
        except (ValueError, FloatingPointError):
            pass
            
    return corr, best_period, best_depth, best_snr

def extract_features(x_ts, stars, cadences, split_name):
    # Vectorized fast math on the CPU
    flux = x_ts[:, 0, :].astype(np.float64) / 1000.0
    col_move = x_ts[:, 1, :].astype(np.float64)
    row_move = x_ts[:, 2, :].astype(np.float64)
    centroid_motion = np.hypot(col_move, row_move)
    
    features = pd.DataFrame({
        'flux_std': np.std(flux, axis=1),
        'flux_skew': stats.skew(flux, axis=1, bias=False, nan_policy='omit'),
        'flux_kurtosis': stats.kurtosis(flux, axis=1, bias=False, nan_policy='omit'),
        'flux_min': np.min(flux, axis=1),
        'flux_max': np.max(flux, axis=1),
        'flux_p01': np.percentile(flux, 1, axis=1),
        'flux_p99': np.percentile(flux, 99, axis=1),
    })

    time = cadences['time_btjd'].to_numpy(dtype=float)
    if len(time) != x_ts.shape[2] or not np.all(np.isfinite(time)):
        raise ValueError(f'{split_name} cadence times are invalid.')

    print(f"{split_name}: Starting parallel BLS processing across all CPU cores...")
    
    # Run the heavy BLS math in parallel
    # n_jobs=-1 tells it to use 100% of your CPU cores
    results = Parallel(n_jobs=-1, backend='loky')(
        delayed(process_single_star)(i, flux[i], centroid_motion[i], time) 
        for i in range(len(stars))
    )
    
    # Unpack results
    corr, best_period, best_depth, best_snr = zip(*results)

    features['centroid_dip_corr'] = corr
    features['bls_best_period'] = best_period
    features['bls_depth'] = best_depth
    features['bls_snr'] = best_snr
    
    meta = stars[META_COLUMNS].copy()
    return pd.concat([features, meta.reset_index(drop=True)], axis=1)

X_tab_train_raw = extract_features(X_ts_train, train_stars, train_cadences, 'train')
X_tab_test_raw = extract_features(X_ts_test, test_stars, test_cadences, 'test')

train: Starting parallel BLS processing across all CPU cores...
test: Starting parallel BLS processing across all CPU cores...


## Dataset and dual-branch residual network

In [7]:
class CustomDataset(Dataset):
    def __init__(self, x_ts, x_meta, labels=None):
        self.x_ts = torch.as_tensor(np.asarray(x_ts, dtype=np.float32))
        self.x_meta = torch.as_tensor(np.asarray(x_meta, dtype=np.float32))
        self.labels = None if labels is None else torch.as_tensor(np.asarray(labels, dtype=np.int64))
        if len(self.x_ts) != len(self.x_meta) or (self.labels is not None and len(self.labels) != len(self.x_ts)):
            raise ValueError('Time-series, metadata, and label row counts must match.')

    def __len__(self):
        return len(self.x_ts)

    def __getitem__(self, index):
        if self.labels is None:
            return self.x_ts[index], self.x_meta[index]
        return self.x_ts[index], self.x_meta[index], self.labels[index]


class ResBlock1D(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size=7, stride=stride, padding=3, bias=False)
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size=5, padding=2, bias=False)
        self.bn2 = nn.BatchNorm1d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.skip = nn.Identity() if stride == 1 and in_channels == out_channels else nn.Sequential(
            nn.Conv1d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
            nn.BatchNorm1d(out_channels),
        )

    def forward(self, x):
        residual = self.skip(x)
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))
        return self.relu(x + residual)

class PlanetResNet(nn.Module):
    def __init__(self, n_meta, n_classes):
        super().__init__()
        self.ts_branch = nn.Sequential(
            nn.Conv1d(3, 32, kernel_size=9, stride=2, padding=4, bias=False),
            nn.BatchNorm1d(32), nn.ReLU(inplace=True), nn.MaxPool1d(kernel_size=3, stride=2, padding=1),
            ResBlock1D(32, 32),
            ResBlock1D(32, 64, stride=2),
            ResBlock1D(64, 128, stride=2),
            nn.AdaptiveAvgPool1d(1), nn.Flatten(),
        )
        self.meta_branch = nn.Sequential(
            nn.Linear(n_meta, 64), nn.BatchNorm1d(64), nn.ReLU(inplace=True), nn.Dropout(0.2),
            nn.Linear(64, 32), nn.ReLU(inplace=True),
        )
        self.head = nn.Sequential(
            nn.Linear(128 + 32, 96), nn.ReLU(inplace=True), nn.Dropout(0.3),
            nn.Linear(96, n_classes),
        )

    def forward(self, x_ts, x_meta):
        return self.head(torch.cat([self.ts_branch(x_ts), self.meta_branch(x_meta)], dim=1))
print(f'Model ready for {len(class_names)} classes and {len(META_COLUMNS) + 11} tabular inputs.')

Model ready for 8 classes and 18 tabular inputs.


## Grouped cross-validation and best-fold checkpoints

Each fold fits its imputer and scaler on that fold's training rows only. Set `EPOCHS` lower for a quick smoke run; the best weights are saved under `checkpoints_1d_resnet/`.

In [8]:
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader

class CustomDataset(Dataset):
    def __init__(self, x_ts, x_meta, labels=None):
        # Keep as NumPy arrays to prevent massive RAM duplication across DataLoader workers
        self.x_ts = x_ts
        self.x_meta = np.asarray(x_meta, dtype=np.float32)
        self.labels = np.asarray(labels, dtype=np.int64) if labels is not None else None
        
        if len(self.x_ts) != len(self.x_meta) or (self.labels is not None and len(self.labels) != len(self.x_ts)):
            raise ValueError('Row counts must match.')

    def __len__(self):
        return len(self.x_ts)

    def __getitem__(self, index):
        # Convert to tensors on-the-fly per item to save memory
        ts = torch.from_numpy(self.x_ts[index].astype(np.float32))
        meta = torch.from_numpy(self.x_meta[index])
        
        if self.labels is None:
            return ts, meta
        return ts, meta, torch.tensor(self.labels[index], dtype=torch.long)

In [9]:
N_SPLITS = 4
EPOCHS = 20
BATCH_SIZE = 128
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE = 5
NUM_WORKERS = 4  # Optimized for parallel loading
CHECKPOINT_DIR = Path('checkpoints_1d_resnet')
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

if len(np.unique(train_groups)) < N_SPLITS:
    raise ValueError(f'GroupKFold({N_SPLITS}) needs at least {N_SPLITS} unique groups.')

gkf = GroupKFold(n_splits=N_SPLITS)
oof_prob = np.zeros((len(y), len(class_names)), dtype=np.float32)
test_fold_prob = []
fold_scores = []
fold_preprocessors = []

def make_loader(dataset, shuffle):
    return DataLoader(
        dataset, 
        batch_size=BATCH_SIZE, 
        shuffle=shuffle, 
        num_workers=NUM_WORKERS,
        pin_memory=DEVICE.type == 'cuda', 
        drop_last=shuffle,  # Only drop the last batch during training, not validation
        persistent_workers=NUM_WORKERS > 0
    )

def predict_probabilities(model, loader):
    model.eval()
    outputs = []
    with torch.inference_mode():
        for batch in loader:
            x_ts, x_meta = batch[:2]
            x_ts = x_ts.to(DEVICE, non_blocking=True)
            x_meta = x_meta.to(DEVICE, non_blocking=True)
            with torch.amp.autocast(device_type=DEVICE.type, enabled=USE_AMP):
                logits = model(x_ts, x_meta)
            outputs.append(torch.softmax(logits.float(), dim=1).cpu().numpy())
    return np.concatenate(outputs, axis=0)

for fold, (train_idx, valid_idx) in enumerate(gkf.split(X_ts_train, y, groups=train_groups), start=1):
    print(f'\n--- Fold {fold}/{N_SPLITS}: train={len(train_idx):,}, valid={len(valid_idx):,} ---')
    
    # 1. Preprocessing (Fitted only on train_idx to prevent leakage)
    imputer = SimpleImputer(strategy='median')
    scaler = StandardScaler()
    x_meta_train = scaler.fit_transform(imputer.fit_transform(X_tab_train_raw.iloc[train_idx])).astype(np.float32)
    x_meta_valid = scaler.transform(imputer.transform(X_tab_train_raw.iloc[valid_idx])).astype(np.float32)
    x_meta_test = scaler.transform(imputer.transform(X_tab_test_raw)).astype(np.float32)
    fold_preprocessors.append((imputer, scaler))

    # 2. Datasets & Loaders
    train_ds = CustomDataset(X_ts_train[train_idx], x_meta_train, y[train_idx])
    valid_ds = CustomDataset(X_ts_train[valid_idx], x_meta_valid, y[valid_idx])
    test_ds = CustomDataset(X_ts_test, x_meta_test)
    
    train_loader = make_loader(train_ds, shuffle=True)
    valid_loader = make_loader(valid_ds, shuffle=False)
    test_loader = make_loader(test_ds, shuffle=False)

    # 3. Model & Loss Initialization
    weights = compute_class_weight(
        class_weight='balanced', classes=np.arange(len(class_names)), y=y[train_idx]
    )
    criterion = nn.CrossEntropyLoss(weight=torch.tensor(weights, dtype=torch.float32, device=DEVICE))
    model = PlanetResNet(n_meta=x_meta_train.shape[1], n_classes=len(class_names)).to(DEVICE)
    
    # --- PYTORCH 2.0 KERNEL COMPILATION FOR RTX 4050 ---
    if DEVICE.type == 'cuda' and hasattr(torch, 'compile'):
        print("Compiling model for Tensor Cores...")
        model = torch.compile(model)
        
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    scaler_amp = torch.amp.GradScaler('cuda', enabled=USE_AMP)
    
    checkpoint_path = CHECKPOINT_DIR / f'best_fold_{fold}.pth'
    best_score, best_epoch, epochs_without_improvement = -np.inf, 0, 0

    # 4. Training Loop
    for epoch in range(1, EPOCHS + 1):
        model.train()
        running_loss = 0.0
        
        for x_ts_batch, x_meta_batch, y_batch in train_loader:
            x_ts_batch = x_ts_batch.to(DEVICE, non_blocking=True)
            x_meta_batch = x_meta_batch.to(DEVICE, non_blocking=True)
            y_batch = y_batch.to(DEVICE, non_blocking=True)
            
            optimizer.zero_grad(set_to_none=True)
            
            with torch.amp.autocast(device_type=DEVICE.type, enabled=USE_AMP):
                logits = model(x_ts_batch, x_meta_batch)
                loss = criterion(logits, y_batch)
                
            scaler_amp.scale(loss).backward()
            scaler_amp.step(optimizer)
            scaler_amp.update()
            running_loss += loss.detach().item() * len(y_batch)

        scheduler.step()
        
        # 5. Validation
        valid_probability = predict_probabilities(model, valid_loader)
        valid_prediction = valid_probability.argmax(axis=1)
        valid_score = f1_score(y[valid_idx], valid_prediction, average='macro')
        mean_loss = running_loss / len(train_ds)
        print(f'Epoch {epoch:02d}/{EPOCHS} | loss={mean_loss:.4f} | val macro-F1={valid_score:.4f}')

        if valid_score > best_score:
            best_score, best_epoch, epochs_without_improvement = valid_score, epoch, 0
            
            # Safely extract weights regardless of whether the model is compiled or not
            state_dict = model._orig_mod.state_dict() if hasattr(model, '_orig_mod') else model.state_dict()
            
            torch.save({
                'model_state_dict': state_dict,
                'fold': fold,
                'epoch': epoch,
                'validation_macro_f1': float(valid_score),
                'class_names': class_names.tolist(),
                'n_meta': int(x_meta_train.shape[1]),
            }, checkpoint_path)
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= PATIENCE:
                print(f'Early stopping; best epoch was {best_epoch}.')
                break

    # 6. Load Best Fold Checkpoint & Predict Test Set
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=True)
    
    if hasattr(model, '_orig_mod'):
        model._orig_mod.load_state_dict(checkpoint['model_state_dict'])
    else:
        model.load_state_dict(checkpoint['model_state_dict'])
        
    valid_probability = predict_probabilities(model, valid_loader)
    oof_prob[valid_idx] = valid_probability
    test_fold_prob.append(predict_probabilities(model, test_loader))
    
    fold_scores.append({
        'fold': fold, 'best_epoch': best_epoch,
        'macro_f1': f1_score(y[valid_idx], valid_probability.argmax(axis=1), average='macro'),
        'checkpoint': str(checkpoint_path),
    })
    print(f'Saved {checkpoint_path} | best val macro-F1={best_score:.4f}')
    
    del model, optimizer, scheduler, train_loader, valid_loader, test_loader
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()

fold_results = pd.DataFrame(fold_scores)
oof_score = f1_score(y, oof_prob.argmax(axis=1), average='macro')
print('\nFold scores:')
display(fold_results)
print(f'Overall out-of-fold macro-F1: {oof_score:.4f}')


--- Fold 1/4: train=13,494, valid=4,506 ---
Compiling model for Tensor Cores...


In file included from /home/shlokvats_/.pyenv/versions/3.12.14/include/python3.12/Python.h:12,
                 from /home/shlokvats_/hackathons/hack4hack/.venv/lib/python3.12/site-packages/triton/backends/nvidia/driver.c:9:
/home/shlokvats_/.pyenv/versions/3.12.14/include/python3.12/pyconfig.h:1874:9: warning: ‘_POSIX_C_SOURCE’ redefined
 1874 | #define _POSIX_C_SOURCE 200809L
      |         ^~~~~~~~~~~~~~~
In file included from /usr/include/x86_64-linux-gnu/bits/libc-header-start.h:33,
                 from /usr/include/stdlib.h:26,
                 from /home/shlokvats_/hackathons/hack4hack/.venv/lib/python3.12/site-packages/triton/backends/nvidia/include/cuda.h:56,
                 from /home/shlokvats_/hackathons/hack4hack/.venv/lib/python3.12/site-packages/triton/backends/nvidia/driver.c:1:
/usr/include/features.h:319:10: note: this is the location of the previous definition
  319 | # define _POSIX_C_SOURCE        202405L
      |          ^~~~~~~~~~~~~~~
W1022 08:47:20.617000 14

Epoch 01/20 | loss=1.8815 | val macro-F1=0.3191
Epoch 02/20 | loss=1.6657 | val macro-F1=0.3631
Epoch 03/20 | loss=1.5750 | val macro-F1=0.4175
Epoch 04/20 | loss=1.5000 | val macro-F1=0.4176
Epoch 05/20 | loss=1.4452 | val macro-F1=0.4589
Epoch 06/20 | loss=1.3804 | val macro-F1=0.5084
Epoch 07/20 | loss=1.3418 | val macro-F1=0.4477
Epoch 08/20 | loss=1.2990 | val macro-F1=0.5096
Epoch 09/20 | loss=1.2578 | val macro-F1=0.4874
Epoch 10/20 | loss=1.2241 | val macro-F1=0.5244
Epoch 11/20 | loss=1.1909 | val macro-F1=0.4748
Epoch 12/20 | loss=1.1584 | val macro-F1=0.5487
Epoch 13/20 | loss=1.1306 | val macro-F1=0.5314
Epoch 14/20 | loss=1.0984 | val macro-F1=0.5463
Epoch 15/20 | loss=1.0820 | val macro-F1=0.5680
Epoch 16/20 | loss=1.0561 | val macro-F1=0.5671
Epoch 17/20 | loss=1.0416 | val macro-F1=0.5787
Epoch 18/20 | loss=1.0207 | val macro-F1=0.5636
Epoch 19/20 | loss=1.0179 | val macro-F1=0.5556
Epoch 20/20 | loss=1.0134 | val macro-F1=0.5809
Saved checkpoints_1d_resnet/best_fold_1.

,fold,best_epoch,macro_f1,checkpoint
0,1,20,0.580888,checkpoints_1d_resnet/best_fold_1.pth
1,2,19,0.603633,checkpoints_1d_resnet/best_fold_2.pth
2,3,19,0.610810,checkpoints_1d_resnet/best_fold_3.pth
3,4,17,0.598204,checkpoints_1d_resnet/best_fold_4.pth


Overall out-of-fold macro-F1: 0.5981


## Four-fold test ensemble and submission

In [10]:
if len(test_fold_prob) != N_SPLITS:
    raise RuntimeError('A test prediction is missing; complete all folds before submitting.')
test_probability = np.mean(np.stack(test_fold_prob, axis=0), axis=0)
test_prediction = label_encoder.inverse_transform(test_probability.argmax(axis=1))
submission = pd.DataFrame({'id': test_stars['id'].to_numpy(), 'label': test_prediction})
submission_path = Path('submission_1d_resnet.csv')
submission.to_csv(submission_path, index=False)
if list(submission.columns) != ['id', 'label'] or len(submission) != len(test_stars):
    raise RuntimeError('Submission schema or row count is invalid.')
print(f'Saved {submission_path.resolve()} with {len(submission):,} predictions.')
print('\nEnsemble predicted-label counts:')
display(submission['label'].value_counts().rename_axis('label').to_frame('count'))
display(submission.head())

Saved /home/shlokvats_/hackathons/hack4hack/submission_1d_resnet.csv with 7,000 predictions.

Ensemble predicted-label counts:


,count
label,
quiet_star,1436
starspots,1251
pulsating_star,920
instrument_glitch,875
flare_star,711
planet,701
background_binary,629
eclipsing_binary,477


,id,label
0,TIC914436805,starspots
1,TIC964940782,background_binary
2,TIC906185835,instrument_glitch
3,TIC927004741,instrument_glitch
4,TIC949385375,background_binary
